# AgentCore Memory Hook을 사용하는 LangGraph (장기 메모리)

## 소개

이 Notebook에서는 LangGraph framework를 사용하는 대화형 AI agent에 Amazon Bedrock AgentCore Memory 기능을 통합하는 방법을 살펴봅니다. 여러 대화 session에 걸쳐 **장기 메모리**를 유지하여 agent가 이전 상호 작용에서 사용자 선호도, 식이 제한, 맥락 정보를 추출하고 기억할 수 있도록 하는 데 중점을 둡니다.

## 튜토리얼 세부 정보

| 항목                | 세부 정보                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 장기 대화형                                                                       |
| Agent 사용 사례     | 영양 도우미                                                                       |
| Agentic Framework   | LangGraph                                                                        |
| LLM model           | Anthropic Claude Haiku 4.5                                                     |
| 튜토리얼 구성 요소  | AgentCore 장기 메모리, Custom Memory Strategies, Pre/Post Model Hooks            |
| 예제 난이도         | 중급                                                                              |

다음 내용을 학습합니다.
- UserPreference custom-override strategy로 AgentCore Memory 생성
- 자동 메모리 저장 및 검색을 위한 pre/post model hook 구현
- 여러 session에 걸쳐 사용자 선호도를 기억하는 영양 도우미 구축
- semantic search를 사용하여 관련 사용자 맥락 검색
- 사용자 지정 메모리 추출 및 통합 prompt 구성

### 시나리오 배경

이 예제에서는 식이 제한, 좋아하는 음식, 요리 선호도, 건강 목표 등 여러 대화에 걸쳐 사용자 맥락을 기억할 수 있는 **영양 도우미**를 만듭니다. Agent는 대화에서 사용자 선호도를 자동으로 추출하여 저장한 뒤, 향후 상호 작용에서 관련 맥락을 검색해 개인화된 영양 조언을 제공합니다.

## 아키텍처

<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## 사전 요구 사항

- Python 3.10+
- 적절한 권한이 있는 AWS account
- AgentCore Memory에 필요한 권한이 있는 AWS IAM role
- Amazon Bedrock model에 대한 액세스

환경을 설정하며 시작해 보겠습니다!

In [ ]:
# https://github.com/langchain-ai/langchain-aws에서 필요한 library 설치
%pip install -qr requirements.txt

In [ ]:
import os
import logging

# LangGraph 및 LangChain component import
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableConfig
from langgraph.store.base import BaseStore
import uuid


region = os.getenv("AWS_REGION", "us-east-1")
logging.getLogger("math-agent").setLevel(logging.DEBUG)

In [ ]:
# Store로 사용할 AgentCoreMemoryStore import
from langgraph_checkpoint_aws import AgentCoreMemoryStore

# 이 예제에서는 맥락을 저장하는 데 InMemorySaver를 사용합니다.
# Production 환경에서는 memory store와 원활하게 연동되는 AgentCoreMemorySaver를 checkpointer로 사용하는 것을 적극 권장합니다.
# from langgraph_checkpoint_aws import AgentCoreMemorySaver
from langgraph.checkpoint.memory import InMemorySaver
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

from custom_memory_prompts import consolidation_prompt, extraction_prompt

In [ ]:
memory_name = "NutritionAssistant"
client = MemoryClient(region_name=region)
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

memory = client.create_or_get_memory(
    name=memory_name,
    description="Nutrition assistant",
    memory_execution_role_arn="arn:aws:iam::YOUR_ACCOUNT:role/YOUR_ROLE",  # 유효한 trust policy가 있는 role을 입력하세요
    strategies=[
        {
            StrategyType.CUSTOM.value: {
                "name": "NutritionPreferences",
                "description": "Captures customer food preferences and behavior",
                "namespaceTemplates": ["/{actorId}/preferences/"],
                "configuration": {
                    "userPreferenceOverride": {
                        "extraction": {
                            "appendToPrompt": extraction_prompt,
                            "modelId": MODEL_ID,
                        },
                        "consolidation": {
                            "appendToPrompt": consolidation_prompt,
                            "modelId": MODEL_ID,
                        },
                    }
                },
            }
        },
    ],
)
memory_id = memory["id"]

### Memory 구성 개요

AgentCore Memory 설정은 다음으로 구성됩니다.

- **Custom Strategy**: 대화에서 영양 관련 선호도 추출
- **Namespaces**: 사용자별로 메모리 구성 (`{actorId}/preferences/`)
- **Custom Prompts**: 음식 선호도에 특화된 추출 및 통합 로직
- **Model Integration**: 메모리 처리에 Claude 3.7 Sonnet 사용

Memory system은 대화를 자동으로 처리하여 일시적이거나 관련 없는 정보는 제외하고 지속적인 사용자 선호도를 추출합니다.

## 3단계: Memory Store 및 LLM 초기화

이제 AgentCore Memory Store와 language model을 초기화합니다.

In [ ]:
# 장기 메모리 저장 및 검색을 활성화하도록 store 초기화
store = AgentCoreMemoryStore(memory_id=memory_id, region_name=region)

# Bedrock LLM 초기화
llm = init_chat_model(MODEL_ID, model_provider="bedrock_converse", region_name=region)

## 4단계: Memory Hook 구현

메모리 저장 및 검색을 자동으로 처리하는 pre/post model hook을 만듭니다.

- **Pre-model hook**: LLM 호출 전에 semantic search를 기반으로 관련 사용자 선호도를 검색하고 맥락 추가
- **Post-model hook**: 장기 메모리 추출을 위해 대화 message 저장

### Memory 처리 방식

1. Message가 actor_id 및 session_id와 함께 AgentCore Memory에 저장됩니다.
2. Custom strategy가 대화를 처리하여 영양 관련 선호도를 추출합니다.
3. 추출된 선호도가 `{actorId}/preferences/` namespace에 저장됩니다.
4. 이후 대화에서 관련 선호도를 검색하고 가져와 맥락으로 활용할 수 있습니다.

**참고**: 장기 메모리로 올바르게 추출될 수 있도록 store 내부에서 LangChain message type을 AgentCore Memory message type으로 변환합니다.

In [ ]:
def pre_model_hook(state, config: RunnableConfig, *, store: BaseStore):
    """최신 사용자 메시지를 저장하기 위해 LLM 호출 전에 실행되는 훅입니다."""
    actor_id = config["configurable"]["actor_id"]
    thread_id = config["configurable"]["thread_id"]
    # Runtime에 전달된 actor와 session 조합에 message 저장
    namespace = (actor_id, thread_id)

    messages = state.get("messages", [])
    # LLM 호출 전에 확인한 마지막 human message 저장
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage):
            store.put(namespace, str(uuid.uuid4()), {"message": msg})
            break
    # 마지막 message를 기반으로 사용자 선호도를 검색하여 state에 추가
    user_preferences_namespace = (actor_id, "preferences/")
    preferences = store.search(user_preferences_namespace, query=msg.content, limit=5)

    # 현재 message 앞에 맥락을 추가할 별도의 AI message 구성
    if preferences:
        context_items = [pref.value for pref in preferences]
        context_message = AIMessage(content=f"[User Context: {', '.join(str(item) for item in context_items)}]")
        # 마지막 human message 앞에 context message 삽입
        return {"messages": messages[:-1] + [context_message, messages[-1]]}

    return {"llm_input_messages": messages}


def post_model_hook(state, config: RunnableConfig, *, store: BaseStore):
    """최신 사용자 메시지를 저장하기 위해 LLM 호출 후에 실행되는 훅입니다."""
    actor_id = config["configurable"]["actor_id"]
    thread_id = config["configurable"]["thread_id"]

    # Runtime에 전달된 actor와 session 조합에 message 저장
    namespace = (actor_id, thread_id)

    messages = state.get("messages", [])
    # LLM 응답을 AgentCore Memory에 저장
    for msg in reversed(messages):
        if isinstance(msg, AIMessage):
            store.put(namespace, str(uuid.uuid4()), {"message": msg})
            break

    return {"messages": messages}

## 5단계: LangGraph Agent 생성

이제 memory hook을 통합한 LangGraph의 `create_react_agent`를 사용하여 영양 도우미 agent를 만듭니다. Tool node에는 장기 메모리 검색 tool만 포함하며, pre/post model hook은 인수로 지정합니다.

**참고**: 사용자 지정 agent 구현에서는 이 패턴을 따르는 모든 workflow에서 필요에 따라 Store와 tool이 실행되도록 구성할 수 있습니다. Pre/post model hook을 사용하거나 마지막에 전체 대화를 저장하는 등의 방식이 가능합니다.

In [ ]:
graph = create_react_agent(
    llm,
    store=store,
    tools=[],  # 이 예제에는 추가 tool이 필요하지 않음
    checkpointer=InMemorySaver(),  # 대화 state 관리용
    pre_model_hook=pre_model_hook,  # LLM 호출 전에 사용자 선호도 검색
    post_model_hook=post_model_hook,  # LLM 응답 후 대화 저장
)

## 6단계: Agent Runtime 구성

사용자와 session을 구분하는 고유 식별자로 agent를 구성해야 합니다. 이 ID는 메모리 구성과 검색에 매우 중요합니다.

### Graph 호출 입력
가장 최근의 user message만 `inputs` 인수로 전달하면 됩니다. 다른 state 변수도 포함할 수 있지만, 간단한 `create_react_agent`에서는 message만 필요합니다.

### LangGraph RuntimeConfig
LangGraph에서 config는 user ID나 session ID처럼 호출 시 필요한 속성을 포함하는 `RuntimeConfig`입니다. `AgentCoreMemorySaver`를 사용하려면 config에 `thread_id`와 `actor_id`를 설정해야 합니다. 예를 들어 AgentCore 호출 endpoint에서 호출자의 identity 또는 user ID를 기반으로 이 값을 할당할 수 있습니다. 자세한 내용은 [여기에서 확인할 수 있습니다](https://langchain-ai.github.io/langgraphjs/how-tos/configuration/).



In [ ]:
actor_id = "user-1"
config = {
    "configurable": {
        "thread_id": "session-1",  # 필수: 내부적으로 Bedrock AgentCore session_id에 매핑됨
        "actor_id": actor_id,  # 필수: 내부적으로 Bedrock AgentCore actor_id에 매핑됨
    }
}

## 7단계: Agent 테스트

음식 선호도에 관한 대화를 통해 영양 도우미를 테스트해 보겠습니다. Agent는 이후에 활용할 수 있도록 사용자 선호도를 자동으로 추출하여 저장합니다.

In [ ]:
# 실행 중 agent 출력을 보기 좋게 표시하는 helper function
def run_agent(query: str, config: RunnableConfig):
    printed_ids = set()
    events = graph.stream(
        {"messages": [{"role": "user", "content": query}]},
        config,
        stream_mode="values",
    )
    for event in events:
        if "messages" in event:
            for msg in event["messages"]:
                # 이 message가 이미 출력되었는지 확인
                if id(msg) not in printed_ids:
                    msg.pretty_print()
                    printed_ids.add(id(msg))


prompt = """
Hey there! Im cooking one of my favorite meals tonight, salmon with rice and veggies (healthy). Has
great macros for my weightlifting competition that is coming up. What can I add to this dish to make it taste better
and also improve the protein and vitamins I get?
"""

run_agent(prompt, config)

### 무엇이 저장되었나요?
보시는 것처럼 model은 아직 사용자의 선호도나 식이 제한에 대한 정보를 가지고 있지 않습니다.

Pre/post model hook을 사용하는 이 구현에서는 두 개의 message가 저장되었습니다. 첫 번째 user message와 AI model의 응답이 모두 AgentCore Memory에 대화 event로 저장되었습니다. 장기 메모리 추출에는 잠시 시간이 걸릴 수 있으므로 처음에 아무것도 검색되지 않으면 몇 초 후 다시 시도하세요.

이후 이 message들은 fact 및 user preferences namespace의 AgentCore 장기 메모리로 추출됩니다. 지금까지 무엇이 저장되었는지 store를 직접 확인해 보겠습니다.

In [ ]:
# User preferences namespace 검색
search_namespace = (actor_id, "preferences/")
result = store.search(search_namespace, query="food", limit=3)
print(f"Preferences namespace result: {result}")

### Agent의 store 액세스

**참고** - AgentCore Memory가 이러한 event를 background에서 처리하므로 메모리가 추출되고 장기 메모리 검색용 embedding이 생성되기까지 몇 초 정도 걸릴 수 있습니다.

좋습니다! 앞선 대화 message를 바탕으로 장기 메모리가 namespace에 추출된 것을 확인했습니다.

이제 새 session을 시작하고 저녁 메뉴 추천을 요청해 보겠습니다. Agent는 store를 통해 추출된 장기 메모리에 액세스하여 사용자가 좋아할 만한 메뉴를 추천할 수 있습니다.

In [ ]:
config = {
    "configurable": {
        "thread_id": "session-2",  # 새로운 session ID
        "actor_id": actor_id,  # 동일한 actor ID
    }
}

run_agent("Today's a new day, what should I make for dinner tonight?", config)

### 마무리

보시는 것처럼 agent는 user preferences namespace 검색을 통해 pre-model hook의 맥락을 전달받았으며, fact namespace에서 장기 메모리를 직접 검색하여 사용자에게 종합적인 답변을 제공할 수 있었습니다.

AgentCoreMemoryStore는 매우 유연하여 pre/post model hook을 사용하거나 store operation을 수행하는 tool만 사용하는 등 다양한 방식으로 구현할 수 있습니다. Checkpointing에 AgentCoreMemorySaver를 함께 사용하면 전체 대화 state와 장기 insight를 결합하여 복잡하고 지능적인 agent system을 구성할 수 있습니다.